# PHASE 3 - Machine Learning Model Development

**Note**: The derived target `High_Value_Transaction` is used for demonstration purposes. The original dataset does not contain a real fraud label.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/Bank_Transactions_Cleaned.csv')


## 1. Feature Selection & 2. Target Creation

In [ ]:
# Selecting the requested features
features = ['CustAccountBalance', 'TransactionTime', 'TransactionAmount (INR)']
X = df[features].copy()

# Create Target: High_Value_Transaction if Amount > 2000 (just a threshold for demonstration)
threshold = 2000
y = (X['TransactionAmount (INR)'] > threshold).astype(int)
print(f"Target 'High_Value_Transaction' created with threshold > {threshold}. Count: {y.sum()}")


## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")


## 4. Missing-Value Handling & 5. Algorithms

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    }

# Create preprocessing pipeline
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

models = {
    'Logistic Regression': Pipeline([('prep', preprocessor), ('clf', LogisticRegression())]),
    'Decision Tree': Pipeline([('prep', preprocessor), ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))]),
    'Random Forest': Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42))])
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test)
    metrics['Model'] = name
    results.append(metrics)
    print(f"--- {name} ---")
    print(classification_report(y_test, model.predict(X_test)))


## 7. Model Comparison

In [ ]:
results_df = pd.DataFrame(results)[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]
display(results_df)
results_df.to_csv('../outputs/tables/model_comparison.csv', index=False)


## 8. Final Model Selection

In [ ]:
# Select Random Forest as the final model based on balanced metrics (usually F1 is best for imbalanced)
final_model = models['Random Forest']
joblib.dump(final_model, '../models/final_model.pkl')
print("Final model saved successfully to '../models/final_model.pkl'.")
